<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/rolling_stats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)
print("folder ready")

import sys
sys.path.append('/content/drive/MyDrive/ml_project')

In [ ]:
%%writefile /content/drive/MyDrive/ml_project/rolling_stats.py
# -*- coding: utf-8 -*-
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

target_dir = "/content/drive/MyDrive/ml_project"
input_pickle = os.path.join(target_dir, "df_lags.pkl")
choice_file = os.path.join(target_dir, "rolling_method.txt")
output_pickle = os.path.join(target_dir, "df_rolling.pkl")

# ---------------------------------------------------------
# rolling_stats_ui
# ---------------------------------------------------------
def rolling_stats_ui():

    if not os.path.exists(input_pickle):
        raise FileNotFoundError("df_lags.pkl not found")

    df = pd.read_pickle(input_pickle)

    label_col = widgets.Label("wybierz kolumnę do obliczeń kroczących:")
    dropdown_col = widgets.Dropdown(options=df.columns.tolist())

    label_win = widgets.Label("podaj długość okna (np. 3, 7, 14):")
    text_win = widgets.Text()

    label_agg = widgets.Label("wybierz rodzaj agregacji (mean, sum, median):")
    dropdown_agg = widgets.Dropdown(options=["mean", "sum", "median"])

    btn = widgets.Button(description="confirm")
    out = widgets.Output()

    def on_click(b):
        out.clear_output()

        col = dropdown_col.value
        win = text_win.value.strip()
        agg = dropdown_agg.value

        with open(choice_file, "w") as f:
            f.write(col + "\n")
            f.write(win + "\n")
            f.write(agg + "\n")

        with out:
            print("saved:", choice_file)
            print("column:", col)
            print("window:", win)
            print("aggregation:", agg)

    btn.on_click(on_click)

    display(widgets.VBox([
        label_col,
        dropdown_col,
        label_win,
        text_win,
        label_agg,
        dropdown_agg,
        btn,
        out
    ]))

# ---------------------------------------------------------
# rolling_stats
# ---------------------------------------------------------
def rolling_stats():

    if not os.path.exists(input_pickle):
        raise FileNotFoundError("df_lags.pkl not found")

    if not os.path.exists(choice_file):
        raise FileNotFoundError("rolling_method.txt not found")

    df = pd.read_pickle(input_pickle)

    with open(choice_file, "r") as f:
        lines = f.read().strip().split("\n")

    col = lines[0]
    win_raw = lines[1]
    agg = lines[2]

    if not win_raw.isdigit():
        raise ValueError("window size must be an integer")

    win = int(win_raw)

    new_col = f"{col}_{agg}_{win}"

    if agg == "mean":
        df[new_col] = df[col].rolling(win).mean()

    elif agg == "sum":
        df[new_col] = df[col].rolling(win).sum()

    elif agg == "median":
        df[new_col] = df[col].rolling(win).median()

    df.to_pickle(output_pickle)

    print("saved:", output_pickle)
    print("generated rolling feature:", new_col)

    return df
